In [33]:
import torch

In [34]:
inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

In [35]:
query = inputs[1] #取出第二个输入词元的词嵌入
#empty创建了一个只有占位的未初始化的张量
attn_scores_2 = torch.empty(inputs.shape[0])
for i,x_i in enumerate(inputs):
    attn_scores_2[i] = torch.dot(x_i,query)

print(attn_scores_2)

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


In [36]:
res = 0

for idx,element in enumerate(inputs[0]):
    res += element * query[idx]

print(res)
print(torch.dot(inputs[0],query))

tensor(0.9544)
tensor(0.9544)


In [37]:
attn_weights_2_tmp = attn_scores_2 / attn_scores_2.sum()

print(attn_weights_2_tmp)
print('Sum:',attn_weights_2_tmp.sum())

tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])
Sum: tensor(1.0000)


In [38]:
def softmax_naive(x):
    return torch.exp(x) / torch.exp(x).sum(dim = 0)

attn_weights_2_naive = softmax_naive(attn_scores_2)

print('Attention weights:',attn_weights_2_naive)
print('Sum:',attn_weights_2_naive.sum())

Attention weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: tensor(1.)


In [39]:
attn_weights_2 = torch.softmax(attn_scores_2,dim = 0)

print('Attention weights:',attn_weights_2)
print('Sum:',attn_weights_2.sum())

Attention weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: tensor(1.)


In [40]:
query = inputs[1]

context_vec_2 = torch.zeros(query.shape)
for i,x_i in enumerate(inputs):
    context_vec_2 += attn_weights_2[i]*x_i

print(context_vec_2)

tensor([0.4419, 0.6515, 0.5683])


In [41]:
attn_scores = torch.empty(6,6)

for i,x_i in enumerate(inputs):
    for j,x_j in enumerate(inputs):
        attn_scores[i,j] = torch.dot(x_i,x_j)

print(attn_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


In [42]:
attn_scores = inputs @ inputs.T
print(attn_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


In [43]:
#dim表示沿着哪一个维度进行归一化,dim = -1表示沿着最内层
    #沿着的意思就是对应维度索引增加的方向
attn_weights = torch.softmax(attn_scores,dim = -1)
print(attn_weights)

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])


In [44]:
row_2_sum = sum([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
print('Row 2 sum:',row_2_sum)
print('All row sums:',attn_weights.sum(dim = -1))

Row 2 sum: 1.0
All row sums: tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])


In [45]:
all_context_vecs = attn_weights @ inputs
print(all_context_vecs)

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])


In [46]:
print('Previous 2nd context vector:',context_vec_2)

Previous 2nd context vector: tensor([0.4419, 0.6515, 0.5683])


In [47]:
x_2 = inputs[1]
d_in = inputs.shape[1]
d_out = 2

In [48]:
torch.manual_seed(123)

#parameter讲张量包装为模型参数
W_query = torch.nn.Parameter(torch.rand(d_in,d_out),requires_grad = False)
W_key   = torch.nn.Parameter(torch.rand(d_in,d_out),requires_grad = False)
W_value = torch.nn.Parameter(torch.rand(d_in,d_out),requires_grad = False)

In [49]:
#区别于注意力权重,这里的是一个可学习参数,前者是一个计算量
query_2 = x_2 @ W_query
key_2 = x_2 @ W_key
value_2 = x_2 @ W_value

print(query_2)

tensor([0.4306, 1.4551])


In [50]:
keys = inputs @ W_key
values = inputs @ W_value

print('keys.shape:',keys.shape)
print('values.shape:',values.shape)

keys.shape: torch.Size([6, 2])
values.shape: torch.Size([6, 2])


In [51]:
keys_2 = keys[1]
attn_score_22 = query_2.dot(keys_2)
print(attn_score_22)

tensor(1.8524)


In [52]:
attn_scores_2 = query_2 @ keys.T
print(attn_scores_2)

tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440])


In [53]:
#工程环境用[-1]
d_k = keys.shape[-1]
#除以d_k**0.5是防止点积太大导致梯度消失
attn_weights_2 = torch.softmax(attn_scores_2 / d_k ** 0.5,dim = -1)
print(attn_weights_2)

tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820])


In [54]:
context_vec_2 = attn_weights_2 @ values
print(context_vec_2)

tensor([0.3061, 0.8210])


In [56]:
import torch.nn as nn

class SelfAttention_v1(nn.Module):
    def __init__(self,d_in,d_out):
        #这里先调用父类的初始化方法
            #super是父类superclass的简写
        super().__init__()
        self.W_query = nn.Parameter(torch.rand(d_in,d_out))
        self.W_key   = nn.Parameter(torch.rand(d_in,d_out))
        self.W_value = nn.Parameter(torch.rand(d_in,d_out))

    def forward(self,x):
        keys = x @ self.W_key
        queries = x @ self.W_query
        values = x @ self.W_value

        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5,dim = -1
        )

        context_vec = attn_weights @ values
        return context_vec

torch.manual_seed(123)
sa_v1 = SelfAttention_v1(d_in,d_out)
print(sa_v1(inputs))

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)


In [60]:
class SelfAttention_v2(nn.Module):
    #bias是b,偏差(偏置)的意思,这里没有
    def __init__(self,d_in,d_out,qkv_bias = False):
        super().__init__()
        #Linear矩阵初始化针对深度学习优化了
            #里面生成一个矩阵,输入一个居住进行矩阵乘法,输入矩阵在前面
        self.W_query = nn.Linear(d_in,d_out,bias = qkv_bias)
        self.W_key   = nn.Linear(d_in,d_out,bias = qkv_bias)
        self.W_value = nn.Linear(d_in,d_out,bias = qkv_bias)

    def forward(self,x):
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5,dim = -1)

        context_vec = attn_weights @ values
        return context_vec

torch.manual_seed(789)
sa_v2 = SelfAttention_v2(d_in,d_out)
#类里面含了call方法,自动导向forward
print(sa_v2(inputs))

tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)


In [61]:
#exercise

In [88]:
qkv_bias = False
W_query = nn.Linear(d_in,d_out,bias = qkv_bias)
W_key   = nn.Linear(d_in,d_out,bias = qkv_bias)
W_value = nn.Linear(d_in,d_out,bias = qkv_bias)

keys = W_key(inputs)
queries = W_query(inputs)
values = W_value(inputs)

attn_scores = queries @ keys.T
attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5,dim = -1)

context_vec = attn_weights @ values
context_vec

tensor([[0.0692, 0.0937],
        [0.0687, 0.0932],
        [0.0688, 0.0933],
        [0.0676, 0.0921],
        [0.0699, 0.0944],
        [0.0669, 0.0913]], grad_fn=<MmBackward0>)

In [90]:
#注意这里提取出来的是转置的形状,需要自己转置一次
test_W_query = W_query.weight.T
test_W_key = W_key.weight.T
test_W_value = W_value.weight.T
print(test_W_query)
queries = inputs @ test_W_query 
keys = inputs @ test_W_key
values = inputs @ test_W_value

attn_scores = queries @ keys.T
attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5,dim = -1)

context_vec = attn_weights @ values
context_vec

tensor([[ 0.5510, -0.5357],
        [-0.4569, -0.2361],
        [ 0.3975,  0.3789]], grad_fn=<PermuteBackward0>)


tensor([[0.0692, 0.0937],
        [0.0687, 0.0932],
        [0.0688, 0.0933],
        [0.0676, 0.0921],
        [0.0699, 0.0944],
        [0.0669, 0.0913]], grad_fn=<MmBackward0>)

In [92]:
queries = sa_v2.W_query(inputs)
keys = sa_v2.W_key(inputs)
attn_scores = queries @ keys.T

attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5,dim = -1)
print(attn_weights)

tensor([[0.1921, 0.1646, 0.1652, 0.1550, 0.1721, 0.1510],
        [0.2041, 0.1659, 0.1662, 0.1496, 0.1665, 0.1477],
        [0.2036, 0.1659, 0.1662, 0.1498, 0.1664, 0.1480],
        [0.1869, 0.1667, 0.1668, 0.1571, 0.1661, 0.1564],
        [0.1830, 0.1669, 0.1670, 0.1588, 0.1658, 0.1585],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<SoftmaxBackward0>)


In [96]:
context_length = attn_scores.shape[0]
#tril的l是low的意思,生成下三角矩阵,下三角保留元素,其余的为0
mask_simple = torch.tril(torch.ones(context_length,context_length))
print(mask_simple)

tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])


In [99]:
#对应元素相乘
masked_simple = attn_weights * mask_simple
print(masked_simple)

tensor([[0.1921, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2041, 0.1659, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2036, 0.1659, 0.1662, 0.0000, 0.0000, 0.0000],
        [0.1869, 0.1667, 0.1668, 0.1571, 0.0000, 0.0000],
        [0.1830, 0.1669, 0.1670, 0.1588, 0.1658, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<MulBackward0>)


In [102]:
#keep_dim保持了维度,哪怕压缩了那么列方向维度依旧是1(如[6,1])
row_sums = masked_simple.sum(dim = -1,keepdim = True)
masked_simple_norm = masked_simple / row_sums
print(masked_simple_norm)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4483, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3800, 0.3097, 0.3103, 0.0000, 0.0000, 0.0000],
        [0.2758, 0.2460, 0.2462, 0.2319, 0.0000, 0.0000],
        [0.2175, 0.1983, 0.1984, 0.1888, 0.1971, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<DivBackward0>)


In [103]:
#默认diagnol是0,保留主对角线元素,这里为1是不保留主对角线元素构成上三角矩阵
mask = torch.triu(torch.ones(context_length,context_length),diagonal = 1)
#加ed表示已经被处理过了
#masked_fill(mask,value) 函数的参数mask必须是bool,所以转换
    #函数吧布尔值为bool为true的地方替换为value的值,false的地方保留原值
masked = attn_scores.masked_fill(mask.bool(),-torch.inf)
print(masked)

tensor([[0.2899,   -inf,   -inf,   -inf,   -inf,   -inf],
        [0.4656, 0.1723,   -inf,   -inf,   -inf,   -inf],
        [0.4594, 0.1703, 0.1731,   -inf,   -inf,   -inf],
        [0.2642, 0.1024, 0.1036, 0.0186,   -inf,   -inf],
        [0.2183, 0.0874, 0.0882, 0.0177, 0.0786,   -inf],
        [0.3408, 0.1270, 0.1290, 0.0198, 0.1290, 0.0078]],
       grad_fn=<MaskedFillBackward0>)


In [104]:
attn_weights = torch.softmax(masked / keys.shape[-1]**0.5,dim = -1)
print(attn_weights)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4483, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3800, 0.3097, 0.3103, 0.0000, 0.0000, 0.0000],
        [0.2758, 0.2460, 0.2462, 0.2319, 0.0000, 0.0000],
        [0.2175, 0.1983, 0.1984, 0.1888, 0.1971, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<SoftmaxBackward0>)


In [107]:
torch.manual_seed(123)
#这是一个类
dropout = torch.nn.Dropout(0.5)
example = torch.ones(6,6)

#补偿不活跃元素,按比例吧其余元素放大了
#Dropout本身就让部分数据变为0,不是Nan
print(dropout(example))

tensor([[2., 2., 0., 2., 2., 0.],
        [0., 0., 0., 2., 0., 2.],
        [2., 2., 2., 2., 0., 2.],
        [0., 2., 2., 0., 0., 2.],
        [0., 2., 0., 2., 0., 2.],
        [0., 2., 2., 2., 2., 0.]])


In [110]:
torch.manual_seed(123)
print(dropout(attn_weights))

tensor([[2.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.7599, 0.6194, 0.6206, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.4921, 0.4925, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.3966, 0.0000, 0.3775, 0.0000, 0.0000],
        [0.0000, 0.3327, 0.3331, 0.3084, 0.3331, 0.0000]],
       grad_fn=<MulBackward0>)


In [111]:
batch = torch.stack((inputs,inputs),dim = 0)
print(batch.shape)

torch.Size([2, 6, 3])


In [114]:
class CausalAttention(nn.Module):
    def __init__(self,d_in,d_out,context_length,
                 dropout,qkv_bias = False):
        super().__init__()
        self.d_out = d_out
        #不开启偏置
        self.W_query = nn.Linear(d_in,d_out,bias = qkv_bias)
        self.W_key = nn.Linear(d_in,d_out,bias = qkv_bias)
        self.W_value = nn.Linear(d_in,d_out,bias = qkv_bias)
        self.dropout = nn.Dropout(dropout)
        #第一个参数是返回的名称,可以用self.mask调取
        #这个函数创建的张量会跟着设备转移
        #context_length是模型支持上下文的上限,不一定等于输入长度
        self.register_buffer('mask',
                             torch.triu(torch.ones(context_length,context_length),
                                        diagonal = 1))

    def forward(self,x):
        b,num_tokens,d_in = x.shape
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        #将批里面的第一维度和第二维度转换,(每个矩阵转置)
        attn_scores = queries @ keys.transpose(1,2)
        #pytorch里面尾随下划线表示就地执行
        attn_scores.masked_fill_(
            #这里切片是为了和实际计算的矩阵匹配
            self.mask.bool()[:num_tokens,:num_tokens],-torch.inf)
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1] ** 0.5,dim = -1
        )
        attn_weights = self.dropout(attn_weights)

        context_vec = attn_weights @ values
        return context_vec

torch.manual_seed(123)
context_length = batch.shape[1]
ca = CausalAttention(d_in,d_out,context_length,0.0)
#call直接调用forward
context_vecs = ca(batch)
print(context_vecs)
print(context_vecs.shape)

tensor([[[-0.4519,  0.2216],
         [-0.5874,  0.0058],
         [-0.6300, -0.0632],
         [-0.5675, -0.0843],
         [-0.5526, -0.0981],
         [-0.5299, -0.1081]],

        [[-0.4519,  0.2216],
         [-0.5874,  0.0058],
         [-0.6300, -0.0632],
         [-0.5675, -0.0843],
         [-0.5526, -0.0981],
         [-0.5299, -0.1081]]], grad_fn=<UnsafeViewBackward0>)
torch.Size([2, 6, 2])


In [115]:
class MultiHeadAttentionWrapper(nn.Module):
    def __init__(self,d_in,d_out,context_length,dropout,num_heads,qkv_bias = False):
        super().__init__()
        self.heads = nn.ModuleList(
            [CausalAttention(d_in,d_out,context_length,dropout,qkv_bias)
             for _ in range(num_heads)]
        )
    def forward(self,x):
        #cat是Concatenate,拼接的意思,沿着dim的维度进行拼接
            #结果是d_out * num_heads的维度
        return torch.cat([head(x) for head in self.heads],dim = -1)


torch.manual_seed(123)
context_length = batch.shape[1]
d_in,d_out = 3,2
mha = MultiHeadAttentionWrapper(
    d_in,d_out,context_length,0.0,num_heads = 2
)

context_vecs = mha(batch)

print(context_vecs)
print(context_vecs.shape)

tensor([[[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]],

        [[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]]], grad_fn=<CatBackward0>)
torch.Size([2, 6, 4])


In [120]:
#exercise 3.2

In [118]:
class MultiHeadAttentionWrapper(nn.Module):
    def __init__(self,d_in,d_out,context_length,dropout,num_heads,qkv_bias = False):
        super().__init__()
        self.heads = nn.ModuleList(
            [CausalAttention(d_in,d_out,context_length,dropout,qkv_bias)
             for _ in range(num_heads)]
        )
    def forward(self,x):
        #cat是Concatenate,拼接的意思,沿着dim的维度进行拼接
            #结果是d_out * num_heads的维度
        return torch.cat([head(x) for head in self.heads],dim = -1)


torch.manual_seed(123)
context_length = batch.shape[1]
d_in,d_out = 3,1
mha = MultiHeadAttentionWrapper(
    d_in,d_out,context_length,0.0,num_heads = 2
)

context_vecs = mha(batch)

print(context_vecs)
print(context_vecs.shape)
dout = 2

tensor([[[-0.5740,  0.2216],
         [-0.7320,  0.0155],
         [-0.7774, -0.0546],
         [-0.6979, -0.0817],
         [-0.6538, -0.0957],
         [-0.6424, -0.1065]],

        [[-0.5740,  0.2216],
         [-0.7320,  0.0155],
         [-0.7774, -0.0546],
         [-0.6979, -0.0817],
         [-0.6538, -0.0957],
         [-0.6424, -0.1065]]], grad_fn=<CatBackward0>)
torch.Size([2, 6, 2])


In [121]:
class MultiHeadAttention(nn.Module):
    def __init__(self,d_in,d_out,context_length,
                 dropout,num_heads,qkv_bias = False):
        #super这是python自带
        super().__init__()
        #这里的断言验证是否可以整除
            #assert <条件> <错误信息>
            #如果条件为True,代码正常通过,如果为False,抛出报错信息
        #\是换行符
        assert (d_out % num_heads == 0), \
            'd_out must be divisible by num_heads'
        self.d_out = d_out
        self.num_heads = num_heads
        #dout是总的输出维度,这里吧多头集合在了一个矩阵里运算了
            #所以这里head_dim才是之前的d_out
        #//得到的int类型
        self.head_dim = d_out // num_heads
        #因为吧多个头集合了,所以一个矩阵够了
        self.W_query = nn.Linear(d_in,d_out,bias = qkv_bias)
        self.W_key = nn.Linear(d_in,d_out,bias = qkv_bias)
        self.W_value = nn.Linear(d_in,d_out,bias = qkv_bias)
        #这是一个可以学习的方阵,可以让输出和其相乘,得到的还是一样的形状,但是每个元素的信息已经交流了
            #避免了之前拼接在一起的数据的生硬
        self.out_proj = nn.Linear(d_out,d_out)
        self.dropout = nn.Dropout(dropout)
        #diagonal有0和1二个模式,0表示包含对角线,1表示不包含对角线
        self.register_buffer(
            'mask',
            torch.triu(torch.ones(context_length,context_length),
                       diagonal = 1)
        )

    def forward(self,x):
        b,num_tokens,d_in = x.shape
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)
        #维度拆解
        keys = keys.view(b,num_tokens,self.num_heads,self.head_dim)
        queries = queries.view(b,num_tokens,self.num_heads,self.head_dim)
        values = values.view(b,num_tokens,self.num_heads,self.head_dim)

        keys = keys.transpose(1,2)
        queries = queries.transpose(1,2)
        values = values.transpose(1,2) 

        attn_scores = queries @ keys.transpose(2,3) 
        mask_bool = self.mask.bool()[:num_tokens,:num_tokens]

        attn_scores.masked_fill_(mask_bool, -torch.inf)

        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1] ** 0.5, dim = -1)
        attn_weights = self.dropout(attn_weights)

        #还原为[b, num_tokens, num_heads, head_dim]
        context_vec = (attn_weights @ values).transpose(1,2)

        #view需要完整的矩阵内存数据,transpose会吧内存数据打乱,这里用contiguous重新排列
        #然后重新压回1D的d_out,不再分头(拼接)
        context_vec = context_vec.contiguous().view(
            b,num_tokens,self.d_out)
        #进行特征融合
        context_vec = self.out_proj(context_vec)
        return context_vec

In [122]:
a = torch.tensor([[[[0.2745, 0.6584, 0.2775, 0.8573],
                    [0.8993, 0.0390, 0.9268, 0.7388],
                    [0.7179, 0.7058, 0.9156, 0.4340]],

                   [[0.0772, 0.3565, 0.1479, 0.5331],
                    [0.4066, 0.2318, 0.4545, 0.9737],
                    [0.4606, 0.5159, 0.4220, 0.5786]]]])

print(a @ a.transpose(2, 3))

tensor([[[[1.3208, 1.1631, 1.2879],
          [1.1631, 2.2150, 1.8424],
          [1.2879, 1.8424, 2.0402]],

         [[0.4391, 0.7003, 0.5903],
          [0.7003, 1.3737, 1.0620],
          [0.5903, 1.0620, 0.9912]]]])


In [123]:
first_head = a[0,0,:,:]
first_res = first_head @ first_head.T
print(first_res)

tensor([[1.3208, 1.1631, 1.2879],
        [1.1631, 2.2150, 1.8424],
        [1.2879, 1.8424, 2.0402]])


In [125]:
second_head = a[0,1,:,:]
#res是result(计算结果的意思)
second_res = second_head @ second_head.T
print(second_res)

tensor([[0.4391, 0.7003, 0.5903],
        [0.7003, 1.3737, 1.0620],
        [0.5903, 1.0620, 0.9912]])


In [128]:
torch.manual_seed(123)
batch_size,context_length,d_in = batch.shape
d_out = 2
mha = MultiHeadAttention(d_in,d_out,context_length,0.0,num_heads = 2)
context_vecs = mha(batch)
print(context_vecs)

tensor([[[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]],

        [[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]]], grad_fn=<ViewBackward0>)


In [130]:
print(context_vecs.shape)

torch.Size([2, 6, 2])


In [131]:
#exercise3.3

In [132]:
torch.manual_seed(123)

context_length = 1024
d_in,d_out = 768,768
mha = MultiHeadAttention(d_in,d_out,context_length,0.0,num_heads = 12)

In [133]:
num_p = 0
#提取可学习参数返回给
for p in mha.parameters():
    #提取元素数量
    num_p += p.numel()

In [136]:
num_p

2360064